## INTRODUCTION
  In Orellana *et al.* 2016, we have used all the crystal structures of a protein to create an ensemble. What we call an ensemble in this article is a set of structures that are not chimeric, represent the whole protein rather than a single domain of it, and do not contain more than 5 consecutive non-terminal missing residues. As much as we would like to automate this process, preparation of an ensemble often requires manual intervention due to problems like, chimeric constructs or the placement of subunits (clockwise vs. anticlockwise). There is also a decision to be made whether to exclude a structure that has non-terminal missing residues.
  
  While preparing the ensembles for this article, we have taken care of those problems manually and made a decision to repair the structures  if it was known to represent an important state which does not have any other representatives. Altough it very much depends on the state structure, and the location of the missing residues, we recommend repairing structures with only less than 5 consecutive residues missing. There is an example modeller script that can be used for this.  It is upto the user to fix and include them in the ensemble later.
  
  This tutorial will be using GLIC as an example. Because this was the most challenging ensemble we prepared, it highlights most of the issues the users are likely to encounter. 

## HOW TO
### Requirements
Before we get started make sure you have installed the pdbParser package, or you are running this notebook in a place where pdbParser is the subfolder. You will find the other required python packages within [github](https://github.com/ozyo/pdbParser) README.md

To determine the missing residues, pdbParser relies on sequence alignment. If you use pdbParser to prepare a start and an end structure for eBDIMs then the alignment is generated via BioPython. However if you are preparing an ensemble you must install Clustal Omega or run the provided fasta sequence through a multiple sequence alignment tool and save the alignment in fasta format. The installation of Clustal Omega can easily be done via conda. After you install it you should provide the full path for the clustal executable.

Altough not a requirement, if you want to fix any broken structures you will need a modelling program. We have provided an example script for MODELLER.

### Obtaining PDB Files
  It can be daunting to find all the available PDB files, and the chains that actually belongs to your protein. The easiset way to do this is to go through the UniProtKB database and pdbParser requires the UniProt ID to prepare the ensemble. 
  ![title](img/uniprot_ini_search.png)
  
After finding the UniProt ID you can run the commands below to gather information. Besides the UniProt ID, you also have to provide the total number of chains that are in a biological assembly. Heteromeric structures require a different routine which are not yet supported in pdbParser. However similar logic applies in preperation of such an ensemble that only the shared CA coordinates should be kept, any broken structures should be fixed or excluded.

You should also provide a folder that exits for inputs/outputs to be read/written. If you use "." This will take the current directory for outputs.

In [1]:
query='Q7NDN8' #UniPortKB ID
mer=5 #Total number of chains
cwd="./" #This directory must exist
#exclude=[List of PDB IDs] 
#If you already know some structures you want to exclude provide them as a list. 

In [2]:
from pdbParser import prepENS as pE
from importlib import reload
reload(pE)
info=pE.PDBInfo(query,mer) #Add the exclude argument to the end.

CRITICAL:root:Cannot process PDB id 3IGQ. It does not contain complete set


PDBInfo function goes into the UniProt page of the given ID, and retrives the 3D structures table (as seen in figure below) and the reference sequence to be used for the alignment. For the reasons below it is much easier to prepare the ensemble via a database like UniProtKB:
* Because there is a possiblity of all the structures having the same missing residues, the reference sequence is a good control to have. 
* Within 3D structure table, only the ones with the label "X-ray" are downloaded. Currently we don't support multi-model pdb files.
* The relevant chains for this protein is also extracted from this table, this is much easier than reading PDB headers.

In the example above, you already see that 3IGQ structure is skipped. That is because this file had 6 subunits vs. 5. If you go into the RCSB Database and look at this structure you will see that it contains only the extracellular domain and not the whole structure. To our advantage this one had more subunits than we required, so it is eliminated early on.

![title](img/uniprot_3dtable.png)

  After the information is gathered, info object is populated with the possible structures to be used and the reference sequence. You can reach the list via commands below. If you want to manipulate this list, add or remove structures, or change the reference sequence you can do so at this stage. Please note that all the functions in prepENS module uses and modifes this class directly.

In [3]:
print(info.query) #UniPort ID used to initialize this class
#and it is also used for the output names.
print(info.mer) # Total number of chains
print(info.result) #Contains the information from the 3D structures table.
print(info.refseq) #Contains the reference sequence. 

Q7NDN8
5
{'2XQ3': [1, [['A', 'B', 'C', 'D', 'E']]], '2XQ4': [1, [['A', 'B', 'C', 'D', 'E']]], '2XQ5': [1, [['A', 'B', 'C', 'D', 'E']]], '2XQ6': [1, [['A', 'B', 'C', 'D', 'E']]], '2XQ7': [1, [['A', 'B', 'C', 'D', 'E']]], '2XQ8': [1, [['A', 'B', 'C', 'D', 'E']]], '2XQ9': [1, [['A', 'B', 'C', 'D', 'E']]], '2XQA': [1, [['A', 'B', 'C', 'D', 'E']]], '3EAM': [1, [['A', 'B', 'C', 'D', 'E']]], '3EHZ': [1, [['A', 'B', 'C', 'D', 'E']]], '3EI0': [1, [['A', 'B', 'C', 'D', 'E']]], '3P4W': [1, [['A', 'B', 'C', 'D', 'E']]], '3P50': [1, [['A', 'B', 'C', 'D', 'E']]], '3TLS': [1, [['A', 'B', 'C', 'D', 'E']]], '3TLT': [1, [['A', 'B', 'C', 'D', 'E']]], '3TLU': [1, [['A', 'B', 'C', 'D', 'E']]], '3TLV': [1, [['A', 'B', 'C', 'D', 'E']]], '3TLW': [1, [['A', 'B', 'C', 'D', 'E']]], '3UU3': [1, [['A', 'B', 'C', 'D', 'E']]], '3UU4': [1, [['A', 'B', 'C', 'D', 'E']]], '3UU5': [1, [['A', 'B', 'C', 'D', 'E']]], '3UU6': [1, [['A', 'B', 'C', 'D', 'E']]], '3UU8': [1, [['A', 'B', 'C', 'D', 'E']]], '3UUB': [2, [['A', 'B', 

  The next command downloads the structures from the RCSB database. Depending on the number of structures this function might take a while to finish because after it downloads the files it goes through the steps below to clean them:
  1. It checks for lines with TITLE to find the chimeric structures.
  2. It seperates the biological assemblies within the same PDB file
  3. It keeps only the CA coordinates, which are the only relevant atoms for comparison with eBDIMs trajectories. Obtaining only CA coordinates is also a way to clean the PDB files from bound ligands, ions, waters, etc.
  4. It extracts the sequence and the residue numbering from the CA coordinates and writes them to text files.

pdbParser relies on multiple sequence alignment to determine the missing residues. This also allows us to ignore the differences in residue numbering between different structures, non-consecutive numbering. The residue numbering information is only used to extract the residues from the the PDB file itself. 
The sequence for the alignment is extracted from CA coordinates, rather than PDB header. Because a residue might have missing CA coordinate, despite being listed in the sequence records.

  In the example below you see that there are more warnings, stating that more structures will be excluded from the ensemble because they are chimeras. The function checks for two keywords in the title 'CHIMERA' and 'FUSED'. Since these structures cannot be used, their information is removed from the info.result dictionary automatically and those files are deleted.

In [4]:
pE.downloadPDB(info,cwd)

[(    1, 'N', '', 'PRO', 'A',   7, '', 29.941, -12.011, 93.546, 1., 136.29, 'N', '')
 (    2, 'CA', '', 'PRO', 'A',   7, '', 29.589, -12.403, 94.92 , 1., 143.07, 'C', '')
 (    3, 'C', '', 'PRO', 'A',   7, '', 28.09 , -12.736, 95.14 , 1., 148.12, 'C', '')
 ...
 (12607, 'CE2', '', 'PHE', 'E', 316, '', 35.96 , -29.252, 34.391, 1., 223.52, 'C', '')
 (12608, 'CZ', '', 'PHE', 'E', 316, '', 36.651, -30.404, 34.069, 1., 223.44, 'C', '')
 (12609, 'OXT', '', 'PHE', 'E', 316, '', 32.007, -29.58 , 37.223, 1., 205.92, 'O', '')]
[(    2, 'CA', '', 'PRO', 'A',   7, '', 29.589, -12.403, 94.92 , 1., 143.07, 'C', '')
 (    9, 'CA', '', 'PRO', 'A',   8, '', 26.105, -14.111, 94.407, 1., 132.17, 'C', '')
 (   16, 'CA', '', 'PRO', 'A',   9, '', 23.292, -14.287, 97.075, 1., 121.78, 'C', '')
 ...
 (12584, 'CA', '', 'PHE', 'E', 314, '', 27.441, -28.013, 36.943, 1., 117.08, 'C', '')
 (12595, 'CA', '', 'GLY', 'E', 315, '', 27.913, -31.398, 35.199, 1., 166.73, 'C', '')
 (12599, 'CA', '', 'PHE', 'E', 316, '', 31.

[(    1, 'N', '', 'PRO', 'A',   7, '', 29.449, -11.66 , 93.915, 1., 134.96, 'N', '')
 (    2, 'CA', '', 'PRO', 'A',   7, '', 28.875, -12.007, 95.22 , 1., 139.47, 'C', '')
 (    3, 'C', '', 'PRO', 'A',   7, '', 27.348, -12.219, 95.198, 1., 149.74, 'C', '')
 ...
 (12602, 'CE2', '', 'PHE', 'E', 316, '', 35.832, -28.928, 35.266, 1., 221.5 , 'C', '')
 (12603, 'CZ', '', 'PHE', 'E', 316, '', 36.463, -29.804, 34.403, 1., 222.61, 'C', '')
 (12604, 'OXT', '', 'PHE', 'E', 316, '', 31.723, -28.814, 37.449, 1., 177.26, 'O', '')]
[(    2, 'CA', '', 'PRO', 'A',   7, '', 28.875, -12.007, 95.22 , 1., 139.47, 'C', '')
 (    9, 'CA', '', 'PRO', 'A',   8, '', 25.402, -13.589, 94.514, 1., 120.18, 'C', '')
 (   16, 'CA', '', 'PRO', 'A',   9, '', 22.493, -13.519, 97.035, 1., 123.25, 'C', '')
 ...
 (12579, 'CA', '', 'PHE', 'E', 314, '', 27.094, -27.426, 37.014, 1.,  82.91, 'C', '')
 (12590, 'CA', '', 'GLY', 'E', 315, '', 27.722, -30.648, 35.08 , 1., 135.29, 'C', '')
 (12594, 'CA', '', 'PHE', 'E', 316, '', 31.

[(    1, 'N', '', 'VAL', 'A',   5, '', 26.672, -8.547, -9.77 , 1., 101.48, 'N', '')
 (    2, 'CA', '', 'VAL', 'A',   5, '', 26.775, -7.983, -8.426, 1., 100.86, 'C', '')
 (    3, 'C', '', 'VAL', 'A',   5, '', 27.94 , -7.003, -8.316, 1., 104.98, 'C', '')
 ...
 (12627, 'CE1', '', 'PHE', 'E', 315, '', 72.059, -6.983, 27.493, 1.,  71.79, 'C', '')
 (12628, 'CE2', '', 'PHE', 'E', 315, '', 73.483, -7.909, 29.166, 1.,  72.33, 'C', '')
 (12629, 'CZ', '', 'PHE', 'E', 315, '', 72.243, -7.82 , 28.575, 1.,  70.87, 'C', '')]
[(    2, 'CA', '', 'VAL', 'A',   5, '', 26.775,  -7.983, -8.426, 1., 100.86, 'C', '')
 (    9, 'CA', '', 'SER', 'A',   6, '', 28.544,  -4.623, -7.858, 1., 100.9 , 'C', '')
 (   15, 'CA', '', 'PRO', 'A',   7, '', 27.873,  -1.591, -5.537, 1., 100.09, 'C', '')
 ...
 (12601, 'CA', '', 'LEU', 'E', 313, '', 80.815, -10.08 , 27.891, 1.,  59.87, 'C', '')
 (12609, 'CA', '', 'PHE', 'E', 314, '', 77.105,  -9.905, 27.046, 1.,  59.22, 'C', '')
 (12620, 'CA', '', 'PHE', 'E', 315, '', 76.512,  

[(    1, 'N', '', 'VAL', 'A',   5, '', 27.926, -9.244, -8.97 , 1., 161.48, 'N', '')
 (    2, 'CA', '', 'VAL', 'A',   5, '', 27.003, -8.157, -8.636, 1., 159.41, 'C', '')
 (    3, 'C', '', 'VAL', 'A',   5, '', 27.784, -6.936, -8.137, 1., 161.65, 'C', '')
 ...
 (12414, 'CE1', '', 'PHE', 'E', 315, '', 71.189, -6.432, 27.108, 1.,  98.73, 'C', '')
 (12415, 'CE2', '', 'PHE', 'E', 315, '', 72.23 , -7.414, 28.986, 1.,  96.4 , 'C', '')
 (12416, 'CZ', '', 'PHE', 'E', 315, '', 71.115, -7.216, 28.224, 1.,  93.56, 'C', '')]
[(    2, 'CA', '', 'VAL', 'A',   5, '', 27.003,  -8.157, -8.636, 1., 159.41, 'C', '')
 (    9, 'CA', '', 'SER', 'A',   6, '', 27.958,  -4.44 , -8.213, 1., 155.6 , 'C', '')
 (   15, 'CA', '', 'PRO', 'A',   7, '', 26.724,  -2.119, -5.346, 1., 145.43, 'C', '')
 ...
 (12388, 'CA', '', 'LEU', 'E', 313, '', 79.839, -10.033, 28.195, 1., 124.32, 'C', '')
 (12396, 'CA', '', 'PHE', 'E', 314, '', 76.181,  -9.786, 27.023, 1., 108.16, 'C', '')
 (12407, 'CA', '', 'PHE', 'E', 315, '', 75.756,  

[(    1, 'N', '', 'VAL', 'A',   5, '', 28.18 , 16.409, -10.164, 1., 101.99, 'N', '')
 (    2, 'CA', '', 'VAL', 'A',   5, '', 27.259, 17.26 ,  -9.407, 1., 101.58, 'C', '')
 (    3, 'C', '', 'VAL', 'A',   5, '', 28.019, 18.434,  -8.766, 1., 104.99, 'C', '')
 ...
 (25218, 'CE1', '', 'PHE', 'J', 315, '', 37.757, 21.595, 183.631, 1.,  72.4 , 'C', '')
 (25219, 'CE2', '', 'PHE', 'J', 315, '', 39.02 , 20.531, 185.353, 1.,  73.32, 'C', '')
 (25220, 'CZ', '', 'PHE', 'J', 315, '', 37.818, 20.741, 184.707, 1.,  71.12, 'C', '')]
[(12607, 'CA', '', 'VAL', 'F',   5, '', -7.116, 22.218, 147.255, 1., 83.25, 'C', '')
 (12614, 'CA', '', 'SER', 'F',   6, '', -5.756, 25.737, 148.169, 1., 82.02, 'C', '')
 (12620, 'CA', '', 'PRO', 'F',   7, '', -6.582, 28.469, 150.818, 1., 80.81, 'C', '')
 ...
 (25192, 'CA', '', 'LEU', 'J', 313, '', 46.265, 17.835, 183.94 , 1., 63.26, 'C', '')
 (25200, 'CA', '', 'PHE', 'J', 314, '', 42.548, 18.25 , 183.074, 1., 57.64, 'C', '')
 (25211, 'CA', '', 'PHE', 'J', 315, '', 42.176, 

...

[(    1, 'N', '', 'VAL', 'A',   5, '', 28.154, -8.735, -9.135, 1.,  95.66, 'N', '')
 (    2, 'CA', '', 'VAL', 'A',   5, '', 27.3  , -7.659, -8.623, 1.,  95.37, 'C', '')
 (    3, 'C', '', 'VAL', 'A',   5, '', 28.126, -6.464, -8.134, 1., 100.63, 'C', '')
 ...
 (12662, 'CE1', '', 'PHE', 'E', 315, '', 73.538, -8.267, 28.957, 1.,  69.19, 'C', '')
 (12663, 'CE2', '', 'PHE', 'E', 315, '', 72.19 , -7.487, 27.153, 1.,  70.29, 'C', '')
 (12664, 'CZ', '', 'PHE', 'E', 315, '', 72.324, -8.224, 28.308, 1.,  67.41, 'C', '')]
[(    2, 'CA', '', 'VAL', 'A',   5, '', 27.3  ,  -7.659, -8.623, 1., 95.37, 'C', '')
 (    9, 'CA', '', 'SER', 'A',   6, '', 28.433,  -4.008, -8.312, 1., 99.33, 'C', '')
 (   15, 'CA', '', 'PRO', 'A',   7, '', 27.589,  -1.688, -5.3  , 1., 94.74, 'C', '')
 ...
 (12636, 'CA', '', 'LEU', 'E', 313, '', 80.766, -10.669, 27.504, 1., 70.16, 'C', '')
 (12644, 'CA', '', 'PHE', 'E', 314, '', 77.014, -10.563, 26.737, 1., 65.85, 'C', '')
 (12655, 'CA', '', 'PHE', 'E', 315, '', 76.585,  -6.82

[(    1, 'N', '', 'VAL', 'A',   5, '', 28.756, -8.631, -9.386, 1., 116.13, 'N', '')
 (    2, 'CA', '', 'VAL', 'A',   5, '', 27.901, -7.543, -8.895, 1., 115.85, 'C', '')
 (    3, 'C', '', 'VAL', 'A',   5, '', 28.735, -6.322, -8.508, 1., 118.44, 'C', '')
 ...
 (12667, 'CE1', '', 'PHE', 'E', 315, '', 73.887, -8.428, 29.207, 1.,  72.6 , 'C', '')
 (12668, 'CE2', '', 'PHE', 'E', 315, '', 72.508, -7.588, 27.435, 1.,  74.18, 'C', '')
 (12669, 'CZ', '', 'PHE', 'E', 315, '', 72.664, -8.38 , 28.56 , 1.,  72.19, 'C', '')]
[(    2, 'CA', '', 'VAL', 'A',   5, '', 27.901,  -7.543, -8.895, 1., 115.85, 'C', '')
 (    9, 'CA', '', 'SER', 'A',   6, '', 28.764,  -3.86 , -8.331, 1., 111.17, 'C', '')
 (   15, 'CA', '', 'PRO', 'A',   7, '', 27.963,  -1.451, -5.447, 1., 101.05, 'C', '')
 ...
 (12641, 'CA', '', 'LEU', 'E', 313, '', 81.06 , -10.699, 27.812, 1.,  73.59, 'C', '')
 (12649, 'CA', '', 'PHE', 'E', 314, '', 77.323, -10.568, 26.986, 1.,  67.78, 'C', '')
 (12660, 'CA', '', 'PHE', 'E', 315, '', 76.951,  

[(    1, 'N', '', 'VAL', 'A',   5, '', 28.334, -8.763, -9.585, 1., 82.12, 'N', '')
 (    2, 'CA', '', 'VAL', 'A',   5, '', 27.544, -7.766, -8.859, 1., 81.73, 'C', '')
 (    3, 'C', '', 'VAL', 'A',   5, '', 28.436, -6.56 , -8.461, 1., 84.78, 'C', '')
 ...
 (12667, 'CE1', '', 'PHE', 'E', 315, '', 73.815, -8.084, 29.481, 1., 41.88, 'C', '')
 (12668, 'CE2', '', 'PHE', 'E', 315, '', 72.5  , -7.238, 27.669, 1., 43.69, 'C', '')
 (12669, 'CZ', '', 'PHE', 'E', 315, '', 72.608, -8.019, 28.805, 1., 41.32, 'C', '')]
[(    2, 'CA', '', 'VAL', 'A',   5, '', 27.544,  -7.766, -8.859, 1., 81.73, 'C', '')
 (    9, 'CA', '', 'SER', 'A',   6, '', 28.563,  -4.094, -8.185, 1., 78.59, 'C', '')
 (   15, 'CA', '', 'PRO', 'A',   7, '', 27.843,  -1.483, -5.391, 1., 71.04, 'C', '')
 ...
 (12641, 'CA', '', 'LEU', 'E', 313, '', 81.168, -10.367, 28.136, 1., 51.37, 'C', '')
 (12649, 'CA', '', 'PHE', 'E', 314, '', 77.415, -10.27 , 27.296, 1., 39.79, 'C', '')
 (12660, 'CA', '', 'PHE', 'E', 315, '', 76.932,  -6.542, 26.

[(    1, 'N', '', 'VAL', 'A',   5, '', 27.898, -7.152, -9.28 , 1., 104.56, 'N', '')
 (    2, 'CA', '', 'VAL', 'A',   5, '', 27.1  , -6.08 , -8.677, 1., 103.84, 'C', '')
 (    3, 'C', '', 'VAL', 'A',   5, '', 27.958, -4.862, -8.33 , 1., 106.75, 'C', '')
 ...
 (12612, 'CE1', '', 'PHE', 'E', 315, '', 73.615, -5.745, 29.166, 1.,  65.63, 'C', '')
 (12613, 'CE2', '', 'PHE', 'E', 315, '', 72.241, -4.95 , 27.391, 1.,  66.56, 'C', '')
 (12614, 'CZ', '', 'PHE', 'E', 315, '', 72.404, -5.73 , 28.511, 1.,  63.14, 'C', '')]
[(    2, 'CA', '', 'VAL', 'A',   5, '', 27.1  , -6.08 , -8.677, 1., 103.84, 'C', '')
 (    9, 'CA', '', 'SER', 'A',   6, '', 28.077, -2.382, -8.28 , 1., 103.35, 'C', '')
 (   15, 'CA', '', 'PRO', 'A',   7, '', 27.224,  0.125, -5.426, 1., 103.52, 'C', '')
 ...
 (12586, 'CA', '', 'LEU', 'E', 313, '', 80.769, -7.756, 28.176, 1.,  70.82, 'C', '')
 (12594, 'CA', '', 'PHE', 'E', 314, '', 77.083, -7.824, 27.084, 1.,  63.5 , 'C', '')
 (12605, 'CA', '', 'PHE', 'E', 315, '', 76.768, -4.084

[(    1, 'N', '', 'VAL', 'D',   5, '', 38.108, -38.831, -14.627, 1., 127.98, 'N', '')
 (    2, 'CA', '', 'VAL', 'D',   5, '', 39.462, -39.007, -15.157, 1., 127.98, 'C', '')
 (    3, 'C', '', 'VAL', 'D',   5, '', 40.157, -40.221, -14.559, 1., 132.88, 'C', '')
 ...
 (12642, 'CE1', '', 'PHE', 'E', 315, '', 72.595,  -5.282,  29.383, 1., 103.21, 'C', '')
 (12643, 'CE2', '', 'PHE', 'E', 315, '', 71.116,  -4.584,  27.646, 1., 102.92, 'C', '')
 (12644, 'CZ', '', 'PHE', 'E', 315, '', 71.355,  -5.32 ,  28.787, 1., 101.44, 'C', '')]
[(    2, 'CA', '', 'VAL', 'D',   5, '', 39.462, -39.007, -15.157, 1., 127.98, 'C', '')
 (    9, 'CA', '', 'SER', 'D',   6, '', 41.204, -42.422, -15.053, 1., 126.32, 'C', '')
 (   15, 'CA', '', 'PRO', 'D',   7, '', 45.061, -42.917, -14.628, 1., 123.69, 'C', '')
 ...
 (12616, 'CA', '', 'LEU', 'E', 313, '', 79.813,  -7.15 ,  28.035, 1.,  97.03, 'C', '')
 (12624, 'CA', '', 'PHE', 'E', 314, '', 76.082,  -7.236,  27.123, 1.,  92.69, 'C', '')
 (12635, 'CA', '', 'PHE', 'E', 3

### Multiple Sequence Alignment and Identification of the Shared Region
If you check the output folder, you will find two text files: "_seq.txt" and "_residmap.txt". We need the information within these files for the multiple sequence alignment and the subsequent cleaning of the terminal residues that are not shared between the structures. It is possible that some structures still contain a terminal tag, that would interfere with the alignment and introduce unncessary gaps. Any chain that contains a non-terminal gap at this stage is marked as broken and the assembly they belong to is skipped.

  If you chose to run the alignment somwhere else, you must clean any headers that is written to it. And the names in the fasta header must be in the same format as in the example file in this tutorial. The alignment should also contain a reference sequence with a header refseq. When you run the function below with the argument alnf="alignmentfile.fasta" , it will only go through the file to determine the broken chains. It does not run clustal, so you don't have to provide it. The alignment file must be placed where *cwd* is set to.

In [8]:
clustalopath="/Users/cattibrie_fr/miniconda2/bin/clustalo"
pE.msa(info,cwd,clustalopath)

CRITICAL:root:3UU3_1.pdb|A| contains insertions or missing residues, skipping this chain and the assembly it belongs to.
CRITICAL:root:3UU3_1.pdb|B| contains insertions or missing residues, skipping this chain and the assembly it belongs to.
CRITICAL:root:3UU3_1.pdb|C| contains insertions or missing residues, skipping this chain and the assembly it belongs to.
CRITICAL:root:3UU3_1.pdb|D| contains insertions or missing residues, skipping this chain and the assembly it belongs to.
CRITICAL:root:3UU3_1.pdb|E| contains insertions or missing residues, skipping this chain and the assembly it belongs to.
CRITICAL:root:4ILA_1.pdb|E| contains insertions or missing residues, skipping this chain and the assembly it belongs to.
CRITICAL:root:4NPQ_2.pdb|H| contains insertions or missing residues, skipping this chain and the assembly it belongs to.
CRITICAL:root:4NPQ_2.pdb|I| contains insertions or missing residues, skipping this chain and the assembly it belongs to.
CRITICAL:root:4NPQ_3.pdb|K| cont

Noticed how I have given only the class instance as an argument? Since I have initialized the input/output filenames and created attributes within this class with the relevant data, I don't need to pass objects individually. So if you need to change names of the sequence file, for example, you can do so by changing the info.seqfilename . If you need to add or delete chains/structures you must modify both the text files and the info.result dictionary.

As you see above there are quite a few chains that have missing residues or insertions. All these identified broken chains are stored in info.broken object. This list is used in the subsequent function that extracts the shared CA region. But before we do that we should take a look at the alignment file. 

Altough the assemblies with broken chains will be skipped, the CA coordinates and the FASTA alignment information of those are kept. For example from the fasta alignment file, we saw that chain S in 4NPQ PDB file, assembly 4 has two missing residues. But the rest of the structures in this assembly is intact. Because there are not many "possible closed state" structures of GLIC, we decided to repair this assembly. You can use the example MODELLER script to rebuild those missing residues. After you have fixed the structure you should follow the steps below:
1. Modify the info.broken object to exclude that chain. 
2. You must also add it back to the info.coreresids and info.coremer.
3. You must rename the final, fixed file to it's original 4NPQ_4.pdb

In [6]:
%run -i 'fix_model.py' #You have to modify the contents of this file if you are working on your own example.
info.broken.remove('4NPQ_4.pdb')
#Python indexing starts from 0
chains=info.result['4NPQ'][1][3] 
#Chain informations are the 2nd element and within that list the 4th ensemble. 
info.coremer['4NPQ_4.pdb']=chains
info.coreresids['4NPQ_4.pdb|S|']=info.coreresids['4NPQ_4.pdb|T|'] 
#Since this is a homopentamer, I have just used the information from other chain to set the residue numbers for |S|

ModuleNotFoundError: No module named 'modeller'

In [9]:
reload(pE)
pE.getcore(info,cwd)

{'4NPQ_3.pdb': ['K', 'L', 'M', 'N', 'O'], '5V6N_1.pdb': ['A', 'B', 'C', 'D', 'E'], '3EAM_1.pdb': ['A', 'B', 'C', 'D', 'E'], '5HEG_1.pdb': ['A', 'B', 'C', 'D', 'E'], '2XQ8_1.pdb': ['A', 'B', 'C', 'D', 'E'], '5MZQ_1.pdb': ['A', 'B', 'C', 'D', 'E'], '2XQ9_1.pdb': ['A', 'B', 'C', 'D', 'E'], '4ILA_1.pdb': ['A', 'B', 'C', 'D', 'E'], '4QH5_1.pdb': ['A', 'B', 'C', 'D', 'E'], '5HEH_1.pdb': ['A', 'B', 'C', 'D', 'E'], '4HFE_1.pdb': ['A', 'B', 'C', 'D', 'E'], '5L4H_1.pdb': ['A', 'B', 'C', 'D', 'E'], '6HZ1_1.pdb': ['A', 'B', 'C', 'D', 'E'], '4HFH_1.pdb': ['A', 'B', 'C', 'D', 'E'], '4ZZB_1.pdb': ['A', 'B', 'C', 'D', 'E'], '5J0Z_1.pdb': ['A', 'B', 'C', 'D', 'E'], '6F12_1.pdb': ['A', 'B', 'C', 'D', 'E'], '5MVN_1.pdb': ['A', 'B', 'C', 'D', 'E'], '6HYV_1.pdb': ['A', 'B', 'C', 'D', 'E'], '3UU3_1.pdb': ['A', 'B', 'C', 'D', 'E'], '2XQ5_1.pdb': ['A', 'B', 'C', 'D', 'E'], '5MZT_1.pdb': ['A', 'B', 'C', 'D', 'E'], '2XQ4_1.pdb': ['A', 'B', 'C', 'D', 'E'], '3TLU_1.pdb': ['A', 'B', 'C', 'D', 'E'], '4HFD_1.pdb': [

Now that we have all the structures we want to use ready, and the list of broken structures is updated we can move on to the next step to extract the common residues. As you can see for GLIC some files does not have the initial valine and serine residues ın the N termini. Some of the structures also have a missing residue in the C termini. The function below, uses the information returned from the msa function above. It finds which residues should be removed from the N and C termini by mapping back the sequence to the residue numbers written to "_residmap.txt". 
The next and the last step is to align the structures.

### Structure Alignment
Since we have extracted a common region, and CA coordinates only, all the structures have the same number of atoms. Regardless of the point mutations and the absolute residue number differences we can use almost any alignment tool to superimpose them. For this step you can use your favorite visualizer. In fact it is a good idea to look at the ensemble, and check each structure individually. For example some GLIC structures have a different subunit placement: anticlockwise vs. clockwise. This makes it impossible to superimpose. To fix this issue you could reorder and rename the chains so that the placement matches the rest of the ensemble.

When you are finished the alignment of the structures, all you need to do is to combine them, then your ensemble is ready for uploading to the server.Since eBDIMs requires MODEL and END lines in the beginning and the end of the file, you can use a bash loop similar to this to combine them. Since all the structures have the same number of atoms, you can do the alignment with this combined file rather then individual files.

In [ ]:
%%bash
touch ../../test/GLIC_ensemble.pdb
for pdb in `ls -1 ../../test/correct*pdb`
do
echo "MODEL" >> ../../test/GLIC_ensemble.pdb
cat $pdb >> ../../test/GLIC_ensemble.pdb
echo "END" >> ../../test/GLIC_ensemble.pdb
done

<table><tr>
<td> <img src="img/side.png" alt="Drawing" style="width: 100%;"/> </td>
<td> <img src="img/top.png" alt="Drawing" style="width: 100%;"/> </td>
</tr></table>

If you have any questions do not hesitate to contact us. pdbParser, ensemble preperation is a package being developed. Watch out for new features on [github](https://github.com/ozyo/pdbParser)